# Expandindo o artigo à RBEF com o pacote `bcs`

Este notebook usa **diretamente** o núcleo de física do simulador (pacote `bcs/`, sem nenhuma dependência de interface gráfica) para reproduzir as figuras e equações do artigo e propor extensões. É um ponto de partida para os próprios estudos futuros mencionados na Seção IV do artigo ("outra atividade... simulador de câmara de bolhas [18]... produção de múons, mésons e bárions estranhos").

Execute com o mesmo Python usado pelo servidor do simulador (`pip install -r requirements.txt`, ou pelo menos `numpy`/`matplotlib` para os gráficos deste notebook).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # raiz do projeto, onde mora o pacote bcs/

import matplotlib.pyplot as plt
import numpy as np

from bcs import bethe_bloch as bb
from bcs import kinematics as kin
from bcs import pair_production as pp
from bcs import event as ev
from bcs.materials import MATERIALS, nucleus_mass_mev

## 1. Reproduzindo a Figura 3 do artigo (perda de energia por ionização)

A Eq. (29) (Bethe-Bloch) é universal em $\beta\gamma = p/Mc$. Aqui comparamos os mesmos materiais da Figura 3 original -- e adicionamos o propano líquido, outro fluido historicamente usado em câmaras de bolhas (citado na Seção II.C do artigo).

In [ ]:
materials_to_plot = ["h2_liquid", "propane_liquid", "he_gas", "carbon", "aluminum", "iron", "tin", "lead"]

fig, ax = plt.subplots(figsize=(7, 5))
for key in materials_to_plot:
    mat = MATERIALS[key]
    curve = bb.dedx_curve(mat, beta_gamma_min=0.1, beta_gamma_max=1e4, n_points=300)
    ax.plot(curve["beta_gamma"], curve["dedx_mev_cm2_g"], label=mat.name_pt)

ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel(r"$\beta\gamma = p/Mc$")
ax.set_ylabel(r"$\langle -dE/dx \rangle$ (MeV cm$^2$ g$^{-1}$)")
ax.set_title("Reprodução da Figura 3 do artigo (+ propano líquido)")
ax.legend(fontsize=8)
ax.grid(True, which="both", alpha=0.25)
plt.show()

**Ideia de extensão:** meça, na fotografia real da Figura 3 do artigo (ou nas do S'Cool Lab), a densidade de bolhas ao longo de um traço de próton em diferentes pontos e compare qualitativamente com esta curva -- essa é exatamente a segunda via de identificação de partículas mencionada na Seção II.C.

## 2. O limiar de produção de pares e a correção de recuo (Eqs. 27-28)

Comparamos o limiar $(E_\gamma)_{\text{mín}}$ para vários núcleos-alvo -- incluindo os presentes nos materiais já cadastrados em `bcs/materials.py` -- com o limite ideal sem recuo $2m_ec^2$.

In [ ]:
for key in ["h2_liquid", "he_gas", "carbon", "iron", "lead"]:
    mat = MATERIALS[key]
    M_N = nucleus_mass_mev(mat)
    E_th = pp.threshold_energy_mev(M_N)
    print(f"{mat.name_pt:30s} M_N={M_N:12.2f} MeV/c^2   (Eg)_min={E_th:.6f} MeV")

print(f"\nLimite sem recuo (M_N -> infinito): {2*0.51099895:.6f} MeV")

**Ideia de extensão:** o artigo observa que, para qualquer núcleo real, $m_e/M_N \sim 10^{-4}$ (ou menor) e a correção de recuo não altera o valor numérico em nenhuma casa decimal relevante. Encontre numericamente qual seria a massa de um "núcleo" hipotético para que a correção de recuo alterasse o limiar em, digamos, 10% -- e compare com massas de partículas reais (um elétron poderia fazer o papel do núcleo?).

## 3. Gerando e inspecionando eventos programaticamente

O mesmo gerador de eventos usado pela interface web pode ser chamado diretamente -- útil para gerar grandes amostras de eventos e estudar distribuições (por exemplo, do momento transverso ou do comprimento de decaimento), algo que a interface interativa não faz por padrão.

In [ ]:
# Distribuição do comprimento de decaimento do K+ (em metros) em funcao do momento,
# usando L = beta*gamma*c*tau (a mesma relacao usada internamente pelo simulador)

from bcs import particles as pdb

kplus = pdb.get("K+")
momenta_gev = np.linspace(0.05, 30, 200)
decay_lengths = [kin.decay_length_m(p*1000, kplus.mass_mev, kplus.mean_lifetime_s) for p in momenta_gev]

plt.figure(figsize=(6,4))
plt.plot(momenta_gev, decay_lengths)
plt.axhline(2.0, color="gray", ls="--", label="câmara de 2 m (diâmetro típico)")
plt.xlabel("momento do K+ (GeV/c)")
plt.ylabel("comprimento médio de decaimento <L> (m)")
plt.title(r"$\langle L \rangle = \beta\gamma c \tau$ para o K$^+$")
plt.legend(); plt.grid(alpha=.3)
plt.show()

print("Momento minimo para <L> > 2 m:", momenta_gev[np.argmax(np.array(decay_lengths) > 2.0)], "GeV/c")

**Ideia de extensão:** repita para outras partículas da tabela (`Lambda0`, `Sigma-`, `pi+`, ...) e discuta por que, no feixe de 24 GeV/c do PS do CERN usado nas fotografias do artigo, certas partículas quase sempre decaem dentro da câmara enquanto outras quase sempre escapam.

In [ ]:
# Gera 500 eventos da reacao K- + p -> Xi- + K+ e histograma o momento do Xi-

momenta = []
for seed in range(500):
    data = ev.generate_cascade_event(reaction_id="k_xi_minus_kplus", beam_momentum_gev=24.0,
                                      B_tesla=1.7, material_key="h2_liquid", seed=seed,
                                      n_background_tracks=0)
    xi_track = next(t for t in data["tracks"] if t["particle"] == "Xi-")
    momenta.append(xi_track["p_start_mev"] / 1000.0)

plt.figure(figsize=(6,4))
plt.hist(momenta, bins=30)
plt.xlabel(r"momento do $\Xi^-$ (GeV/c)")
plt.ylabel("eventos")
plt.title(r"Distribuição de momento do $\Xi^-$ em $K^- p \to \Xi^- K^+$ (24 GeV/c)")
plt.grid(alpha=.3)
plt.show()

## 4. Próximos passos sugeridos (ligados à Seção IV do artigo)

- **Produção de múons e mésons/bárions estranhos a partir do decaimento de píons**, como sugerido no parágrafo final do artigo: já está parcialmente implementado (`pi+ -> mu+`, `K- -> mu-`, ...) na tabela `bcs/particles.py` -- basta seguir a cadeia de decaimentos nos eventos de cascata.
- **Leis de conservação em partículas compostas** (número bariônico $B$, número leptônico $L$, estranheza $S$): use `tests/test_particles_and_reactions.py` como modelo para verificar, para qualquer reação nova que você adicionar em `bcs/reactions.py`, que $B$ e $Q$ se conservam (e que $S$ só se conserva nas reações fortes, não nos decaimentos fracos).
- **Interações hadrônicas antes da produção de pares** (nota de rodapé da Seção III.E do artigo): a origem hadrônica dos fótons que geram os pares $e^+e^-$ nas fotografias reais (via $\pi^0 \to \gamma\gamma$) já é reproduzida automaticamente no modo "Cascata" do simulador -- inspecione `bcs/event.py:_handle_photon` para estender a probabilidade de conversão ou adicionar novos mésons neutros.
- **Metrologia**: adicione ruído/erro de medida realista à ferramenta de medição (`bcs/measurement.py`) e quantifique como a incerteza no raio medido se propaga para o momento estimado -- um exercício direto de propagação de incertezas para a disciplina de Física Experimental.